# Tâche 3 : Question-réponse avec affinage par instructions du modèle pré-entraîné 

Cette tâche consiste à affiner par instructions le modèle pré-entraîné de la tâche 2 et d'évaluer la qualité de ses réponses. Vous utilisez les mêmes paires **question-réponse** que celles des tâches précentes. Le fichier d'**instructions** pour l'affinage du modèle est également fourni dans le répertoire *data*. 

> Il est recommandé de faire ce travail pratique en utilisant une carte graphique GPU compatible avec HuggingFace/Pytorch.
> Si votre machine n’en possède pas, vous pouvez utiliser **Google Colab** pour exécuter le *notebook* dans le cloud.

In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    pipeline,
    DataCollatorForLanguageModeling,
)
from transformers.pipelines.pt_utils import KeyDataset
from datasets import load_dataset
import torch
import json
import os
import time

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement du modèle Hugging Face et du tokenizer

In [2]:
model = AutoModelForCausalLM.from_pretrained("saved_models/pythia-410m-final-t2")
tokenizer = AutoTokenizer.from_pretrained("saved_models/pythia-410m-final-t2")

if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 1606.44it/s]


In [3]:
inputs = tokenizer("GPT is a ", return_tensors="pt", max_new_tokens=50, max_length=50)
tokens = model.generate(**inputs, repetition_penalty=1.2)
print(tokenizer.decode(tokens[0]))

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\transformers\generation\utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=25) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


GPT is a  model that has been trained on the CIFAR-10 dataset. It was released in June


## 2. Fonctions de test question-réponse 

In [4]:
def build_prompt(question, context=None):
    few_shot = (
        "Question: What mechanism allows a Transformer model to weigh the importance of different words in a sequence?\n"
        "Answer: The self-attention mechanism allows a Transformer to weigh the importance of different tokens relative to each other.\n\n"
        "Question: What is the purpose of positional encodings in the Transformer architecture?\n"
        "Answer: Positional encodings inject information about the relative or absolute position of tokens in the sequence.\n\n"
    )
    if context:
        return f"Answer concisely and factually.\n\n{few_shot}Context: {context}\nQuestion: {question}\nAnswer:"
    return f"Answer concisely and factually.\n\n{few_shot}Question: {question}\nAnswer:"

def clean_generation(generated_text):
    text = generated_text.strip()
    for stop_token in ["\nQuestion:", "\nQ:", "\n###", "\n\n"]:
        if stop_token in text:
            text = text.split(stop_token)[0].strip()
    return text

In [5]:
def to_prompt_completion(example):
    instruction = example["instruction"].strip()
    inp = example.get("input", "").strip()
    output = example["output"].strip()

    question = f"{instruction}\n{inp}" if inp else instruction

    return {
        "prompt": f"Question: {question}\nAnswer:",
        "completion": f" {output}{tokenizer.eos_token}"
    }



## 3. Affinage du modèle par instructions

In [6]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/alpaca_data.json", split="train")

#dataset = dataset.train_test_split(test_size=0.1)

dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names
)


Map: 100%|██████████| 7000/7000 [00:00<00:00, 44819.08 examples/s]


In [7]:
from trl import SFTConfig

output_dir = "./saved_models/pythia-410m-final-t3"

training_args = SFTConfig(
    num_train_epochs=1,                 # 1 à 3 époques recommandées
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,      # Taille de batch effective = 16
    learning_rate=2e-5,                 #
    warmup_steps=0.1,
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    eval_strategy="no",
    eval_steps=50,
    save_strategy="epoch",
    completion_only_loss=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    max_grad_norm=1.0,
    bf16=True
)

In [8]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

Truncating train dataset: 100%|██████████| 7000/7000 [00:00<00:00, 29048.84 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 7000/7000 [00:00<00:00, 143734.78 examples/s]


In [9]:
batch = next(iter(trainer.get_train_dataloader()))
print(batch["labels"][0])

tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, 28429, 12003,
        16038,   310,   253,  8327,   281,   513,  1633,   323,   697,  1211,
        13232,    13,  1223, 38988, 16038,   310,   253,  8327,   281,   513,
         1633,   281,  6351,   247, 10921,   390,  3693, 14232,    15,     0,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, 

In [10]:
trainer.train()

Step,Training Loss
10,1.879034
20,1.948951
30,1.917513
40,1.754021
50,1.718180
60,1.737727
70,1.716684
80,1.713122
90,1.783345
100,1.706755


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.78s/it]


TrainOutput(global_step=438, training_loss=1.7394279301438702, metrics={'train_runtime': 167.3247, 'train_samples_per_second': 41.835, 'train_steps_per_second': 2.618, 'total_flos': 2122375864811520.0, 'train_loss': 1.7394279301438702, 'entropy': 1.7792710661888123, 'mean_token_accuracy': 0.6011808440089226, 'num_tokens': 457222.0, 'epoch': 1.0})

In [11]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s]


('./saved_models/pythia-410m-final-t3\\tokenizer_config.json',
 './saved_models/pythia-410m-final-t3\\tokenizer.json')

In [12]:
inputs = tokenizer(""" "instruction": "Provide five examples of words that use the '-ous' suffix.",""", return_tensors="pt", max_length=24)
tokens = model.generate(**inputs)
print(tokenizer.decode(tokens[0]))

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\transformers\generation\utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=38) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in GPTNeoXLayer. Setting `layer_past=None`.


 "instruction": "Provide five examples of words that use the '-ous' suffix.",
GroGroGroGroGroGroGroGroGroGroGroGroGroGroGroGroGroGroGro


## 4. Génération des réponses aux questions avec le modèle

In [13]:
test_fn = "data/test_questions.json"

raw_dataset = load_dataset("json", data_files=test_fn)["train"]

dataset = raw_dataset.map(
    lambda example: {"prompt": build_prompt(example["question"])},
)
print(f"Nombre de questions chargées dans le Dataset : {len(dataset)}")

Nombre de questions chargées dans le Dataset : 50


In [14]:
pipe_dataset = KeyDataset(dataset, "prompt")

raw_outputs = []
start_time = time.time()

if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

for out in generator(
    pipe_dataset,
    batch_size=16,
    max_new_tokens=264,
    do_sample=True,
    repetition_penalty=1.05,
    top_p=0.7,
    temperature=0.5,
    stop_strings=["\n", "Question:"],
    tokenizer=tokenizer,
    return_full_text=False
):
    raw_outputs.append(out[0]["generated_text"])

print(f"Génération en batch complétée en {time.time() - start_time:.2f} secondes !")

results = []
for sample, raw_gen in zip(dataset, raw_outputs):
    results.append({
        "Q": sample["question"],
        "A": clean_generation(raw_gen),
        "Expected": sample.get("answer", "")
    })

out_dir = "result"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "t3_results.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Fichier de résultats sauvegardé ({len(results)} entrées) : {out_path}")

Génération en batch complétée en 21.90 secondes !
Fichier de résultats sauvegardé (50 entrées) : result\t2_results.json


 ## 5. Évaluation des résultats

In [15]:
import re
from collections import Counter
import evaluate
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


def tokenize_no_stopwords(text: str):
    tokens = re.findall(r"\b\w+\b", text.lower())
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS]


def compute_word_metrics(results: list[dict]):
    precisions, recalls, f1s = [], [], []

    for item in results:
        pred_tokens = tokenize_no_stopwords(item["A"])
        ref_tokens = tokenize_no_stopwords(item["Expected"])

        common = Counter(pred_tokens) & Counter(ref_tokens)
        num_same = sum(common.values())

        if len(pred_tokens) == 0 or len(ref_tokens) == 0 or num_same == 0:
            precisions.append(0.0)
            recalls.append(0.0)
            f1s.append(0.0)
            continue

        p = num_same / len(pred_tokens)
        r = num_same / len(ref_tokens)
        f1 = (2 * p * r) / (p + r)

        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

    n = len(results)
    avg_p = (sum(precisions) / n) * 100 if n > 0 else 0.0
    avg_r = (sum(recalls) / n) * 100 if n > 0 else 0.0
    avg_f1 = (sum(f1s) / n) * 100 if n > 0 else 0.0
    return avg_p, avg_r, avg_f1


squad = evaluate.load("squad")

squad_predictions = [{"id": str(i), "prediction_text": item["A"]} for i, item in enumerate(results)]
squad_references = [{"id": str(i), "answers": {"text": [item["Expected"]], "answer_start": [0]}} for i, item in enumerate(results)]

metrics = squad.compute(predictions=squad_predictions, references=squad_references)
word_precision, word_recall, word_f1 = compute_word_metrics(results)

print("=" * 60)
print(" RÉSULTATS EVALUATE:")
print(f" Exact Match (SQuAD) : {metrics['exact_match']:.2f}%")
print(f" F1-Score (SQuAD)    : {metrics['f1']:.2f}%")
print("-" * 60)
print(f" Précision (Mots)    : {word_precision:.2f}%")
print(f" Rappel (Mots)       : {word_recall:.2f}%")
print(f" F1-Score (Mots)     : {word_f1:.2f}%")
print("=" * 60)

 RÉSULTATS EVALUATE:
 Exact Match (SQuAD) : 0.00%
 F1-Score (SQuAD)    : 4.82%
------------------------------------------------------------
 Précision (Mots)    : 1.90%
 Rappel (Mots)       : 1.74%
 F1-Score (Mots)     : 1.66%


In [16]:
print("Exemples de réponses générées :")
for sample in results[:3]:
    print("-" * 50)
    print(f"Q        : {sample['Q']}")
    print(f"Expected : {sample['Expected']}")
    print(f"A (gen)  : {sample['A']}")

Exemples de réponses générées :
--------------------------------------------------
Q        : What is the context window size of GPT-3?
Expected : GPT-3 has a context window size of 2,048 tokens.
A (gen)  : The World Warrickeditionally, as well as well as well as well-related to the first-based on the same-thesis a few years ago, and itsy, the same-to beaming it washeritage.
--------------------------------------------------
Q        : Explain what is a transformer architecture.
Expected : The transformer architecture is a neural network based on a deep learning model that was introduced in 2017, used in natural language processing (NLP) systems.
A (gen)  : A-based on the same-up to be able to bearer, as well-the only a few years agoing the same-to beacross the first classically, and the same-likewise, as well-in the same-based on the same-in the same-to be-therapid the last yearning the same- the same-like the following the following the same-a.
---------------------------------------